# Transform and validate visitor data

This notebook performs the **transformation** and **validation** stages of the data workflow.

**Objectives**

- Load cleaned tabular data
- Transform to a quarterly, tidy structure suitable for downstream analysis
- Persist interim and processed artefacts
- Run schema and rule-based validations, and surface a concise report

## 0. Setup

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

# Ensure the project src/ is on the Python path for imports when running the notebook directly
PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT / 'src').exists() and str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# Display options for easier inspection
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)

# Paths (prefer config if available)
try:
    from config import DATA_DIR  # src/config.py
    DATA_DIR = Path(DATA_DIR)
    RAW_DIR = DATA_DIR / 'raw'
    INTERIM_DIR = DATA_DIR / 'interim'
    PROCESSED_DIR = DATA_DIR / 'processed'
except Exception:
    DATA_DIR = PROJECT_ROOT / 'data'
    RAW_DIR = DATA_DIR / 'raw'
    INTERIM_DIR = DATA_DIR / 'interim'
    PROCESSED_DIR = DATA_DIR / 'processed'

RAW_FILE = RAW_DIR / 'overseas-visitors-to-britain-2024.xlsx'
INTERIM_OUT = INTERIM_DIR / 'visitors_by_quarter.csv'
PROCESSED_OUT = PROCESSED_DIR / 'visitors_quarterly.parquet'

# Create output folders if missing
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Raw file:', RAW_FILE)
print('Interim out:', INTERIM_OUT)
print('Processed out:', PROCESSED_OUT)

## 1. Import pipeline modules

In [ ]:
# Import project modules
from data import load, clean, transform, validate, schemas

# Optional utility layer for IO if present
try:
    from utils import io as io_utils
except Exception:
    io_utils = None

print('Modules imported: load, clean, transform, validate, schemas')

## 2. Load raw data

In [ ]:
# Read from the ONS Excel workbook.
# If load module exposes a specific function, prefer that; fallback to pandas.
try:
    df_raw = load.read_excel(RAW_FILE)
except AttributeError:
    # Generic fallback: first sheet
    df_raw = pd.read_excel(RAW_FILE)

print(df_raw.shape)
df_raw.head()

## 3. Clean and standardise

In [ ]:
# Apply cleaning rules defined in src/data/clean.py
try:
    df_clean = clean.clean_visitors(df_raw)
except AttributeError:
    # Fallback: standardise column names minimally
    df_clean = df_raw.copy()
    df_clean.columns = [str(c).strip().lower().replace(' ', '_') for c in df_clean.columns]

df_clean.info()
df_clean.head()

## 4. Transform to quarterly tidy dataset

In [ ]:
# Aggregate/reshape to a tidy quarterly table
try:
    df_quarterly = transform.aggregate_quarterly(df_clean)
except AttributeError:
    # Fallback: assume already quarterly; ensure expected dtypes
    df_quarterly = df_clean.copy()

# Quick sense-check: show the latest few quarters
df_quarterly.sort_values(by=df_quarterly.columns[:2].tolist()).tail(8)

### Optional: quick visual check

In [ ]:
# Plot total visits by quarter if the relevant columns exist
cols = [c.lower() for c in df_quarterly.columns]
possible_quarter_cols = ['quarter', 'period', 'qtr', 'year_quarter']
possible_visits_cols = ['visits', 'number_of_visits', 'visits_thousands']
q_col = next((c for c in df_quarterly.columns if c.lower() in possible_quarter_cols), None)
v_col = next((c for c in df_quarterly.columns if c.lower() in possible_visits_cols), None)
if q_col and v_col:
    ax = df_quarterly[[q_col, v_col]].set_index(q_col).sort_index()[v_col].plot(kind='line', figsize=(9, 4), title='Visits by quarter')
    ax.set_xlabel('Quarter')
    ax.set_ylabel('Visits')
else:
    print('Skipping chart: could not infer quarter/visits columns')

## 5. Persist interim and processed artefacts

In [ ]:
# Write CSV (interim)
try:
    if io_utils is not None and hasattr(io_utils, 'write_csv'):
        io_utils.write_csv(df_quarterly, INTERIM_OUT)
    else:
        df_quarterly.to_csv(INTERIM_OUT, index=False)
    print(f'Wrote: {INTERIM_OUT}')
except Exception as e:
    print('Failed to write interim CSV:', e)

# Write Parquet (processed)
try:
    if io_utils is not None and hasattr(io_utils, 'write_parquet'):
        io_utils.write_parquet(df_quarterly, PROCESSED_OUT)
    else:
        df_quarterly.to_parquet(PROCESSED_OUT, index=False)
    print(f'Wrote: {PROCESSED_OUT}')
except Exception as e:
    print('Failed to write processed Parquet:', e)

## 6. Validation

In [ ]:
# Schema and business-rule checks
validation_summary = {}

# 6.1 Schema validation
try:
    quarterly_schema = getattr(schemas, 'QUARTERLY_VISITORS_SCHEMA')
except Exception:
    quarterly_schema = None

try:
    if quarterly_schema is not None and hasattr(validate, 'validate_schema'):
        schema_report = validate.validate_schema(df_quarterly, quarterly_schema)
    elif hasattr(validate, 'validate_quarterly'):
        schema_report = validate.validate_quarterly(df_quarterly)
    else:
        schema_report = {'status': 'skipped', 'detail': 'No schema validator available'}
    validation_summary['schema'] = schema_report
except Exception as e:
    validation_summary['schema'] = {'status': 'error', 'error': str(e)}

# 6.2 Basic rule checks (non-negative metrics, no duplicate quarters per group, etc.)
basic_checks = {}
try:
    # Example checks: replace column names as appropriate in your codebase
    non_negative_cols = [c for c in df_quarterly.columns if any(k in c.lower() for k in ['visit', 'expenditure', 'nights'])]
    negatives = {}
    for c in non_negative_cols:
        nneg = int((df_quarterly[c] < 0).sum()) if pd.api.types.is_numeric_dtype(df_quarterly[c]) else 0
        if nneg:
            negatives[c] = nneg
    basic_checks['non_negative_fail_counts'] = negatives

    # Duplicate key rows check: attempt to infer key columns
    key_candidates = ['year', 'quarter', 'period', 'region', 'country', 'geography', 'purpose', 'transport']
    keys = [c for c in df_quarterly.columns if c.lower() in key_candidates]
    dup_count = int(df_quarterly.duplicated(subset=keys).sum()) if keys else 0
    basic_checks['duplicate_rows_on_keys'] = dup_count

    validation_summary['basic_rules'] = {'status': 'ok' if not negatives and dup_count == 0 else 'warn', 'detail': basic_checks}
except Exception as e:
    validation_summary['basic_rules'] = {'status': 'error', 'error': str(e)}

validation_summary

## 7. Assert pass criteria

In [ ]:
# Define minimal pass criteria; adapt to your schema/rules as required
status = {k: v.get('status', 'unknown') for k, v in validation_summary.items()}
print('Validation status by category:', status)

# Treat any explicit 'error' as a hard failure
if any(v == 'error' for v in status.values()):
    raise AssertionError('Validation errors detected. Inspect validation_summary above.')

print('Validation completed.')

## 8. Save validation report (optional)

In [ ]:
# Optionally persist a JSON validation report alongside artefacts
report_path = PROCESSED_DIR / 'validation_report.json'
try:
    with open(report_path, 'w', encoding='utf-8') as f:
        json.dump(validation_summary, f, indent=2, ensure_ascii=False)
    print(f'Wrote: {report_path}')
except Exception as e:
    print('Skipped writing validation report:', e)

## 9. Next steps

- Use the processed dataset for further analysis (e.g., `src/analysis/summary.py`).
- Produce charts for reporting (e.g., `src/analysis/charts.py`).
- Integrate this notebook into automated checks if desired.